# WFP Access Architecture: Basic Tests

This notebook is deliberately small. It checks that the draft architecture behaves like a lightweight shared-data runtime: one dataset identity is registered once, source and target roles can point to the same handle, artifact calls are cached by contract, and routing strategies can be swapped without changing data objects.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Any

import pandas as pd

PROJECT_ROOT = Path(r"C:\local\GIT\Public-Infrastructure-Service-Access\Research-Sandbox\drafts\WFP")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wfp_access.artifacts.decorators import derived_artifact, source_artifact
from wfp_access.artifacts.spec import ArtifactSpec
from wfp_access.core.context import DataContext
from wfp_access.data.registry import DatasetHandle, DatasetKey
from wfp_access.data.schemas import DistanceMatrixSchema, PointSchema
from wfp_access.routing.base import RouterCapabilities, RouterStrategy
from wfp_access.storage.repository import Repository


## Minimal In-Memory Repository

The production repository can write Parquet/CSV/XLSX/GeoPackage through codecs. For these tests we only need the same interface, backed by a dictionary, so the assertions focus on architecture behavior rather than file formats.


In [ ]:
class InMemoryRepository(Repository):
    def __init__(self, root: Path):
        super().__init__(root=root, default_format="memory")
        self.objects: dict[str, Any] = {}

    def artifact_uri(self, spec: ArtifactSpec, fingerprint: str) -> str:
        return f"memory://{spec.name}/{fingerprint}"

    def exists(self, uri: str | None) -> bool:
        return bool(uri) and uri in self.objects

    def read_dataset(self, key: DatasetKey, schema=None) -> DatasetHandle:
        if key.uri is None:
            raise ValueError("Memory reads need a URI")
        return DatasetHandle(key=key, data=self.objects[key.uri], schema=schema)

    def write_dataset(self, handle: DatasetHandle, spec: ArtifactSpec) -> None:
        if handle.key.uri is None:
            raise ValueError("Memory writes need a URI")
        self.objects[handle.key.uri] = handle.data


ctx = DataContext(
    run_id="basic_architecture_tests",
    repository=InMemoryRepository(PROJECT_ROOT / "data" / "runs" / "basic_architecture_tests"),
)
ctx


## Same Dataset, Two Roles, One Handle

In [ ]:
shared_locations = pd.DataFrame(
    {
        "ID": ["A", "B", "C"],
        "lat": [49.6116, 49.6150, 49.6200],
        "lon": [6.1319, 6.1200, 6.1400],
        "population": [100, 250, 80],
    }
)

shared_key = DatasetKey(
    name="shared_locations",
    uri="memory://source_tables/shared_locations",
    schema=PointSchema.name,
    crs="EPSG:4326",
)

source_handle = ctx.data.register(DatasetHandle(key=shared_key, data=shared_locations, schema=PointSchema))
target_handle = ctx.data.register(DatasetHandle(key=shared_key, data=shared_locations, schema=PointSchema))

assert source_handle is target_handle
assert source_handle.frame() is target_handle.frame()
assert ctx.data.get(shared_key) is source_handle

{
    "same_handle_for_source_and_target": source_handle is target_handle,
    "same_dataframe_object": source_handle.frame() is target_handle.frame(),
    "registry_size": len(ctx.data._cache),
}


## Artifact Cache Identity

In [ ]:
@derived_artifact(name="basic_prepared_points", schema=PointSchema, format_role="table")
def prepare_points(points: DatasetHandle) -> pd.DataFrame:
    frame = points.frame().copy()
    frame["role_ready"] = True
    return frame

prepared_a = ctx.runner.run(prepare_points, source_handle)
prepared_b = ctx.runner.run(prepare_points, source_handle)

assert prepared_a is prepared_b
assert prepared_a.frame().equals(prepared_b.frame())
assert prepared_a.key == prepared_b.key

{
    "same_artifact_handle": prepared_a is prepared_b,
    "artifact_uri": prepared_a.key.uri,
    "artifact_rows": len(prepared_a.frame()),
}


## Interchangeable Routing Strategies

In [ ]:
class ConstantRouter:
    name = "constant"
    contract_version = "0.1"
    capabilities = RouterCapabilities(distance=True)

    def prepare(self, network=None, config=None):
        return self

    def snap(self, points):
        return points

    def route_many(self, origins: pd.DataFrame, destinations: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame(
            [
                {"origin_id": o.ID, "destination_id": d.ID, "distance_m": 1.0}
                for o in origins.itertuples()
                for d in destinations.itertuples()
            ]
        )


class ManhattanRouter(ConstantRouter):
    name = "manhattan"

    def route_many(self, origins: pd.DataFrame, destinations: pd.DataFrame) -> pd.DataFrame:
        rows = []
        for o in origins.itertuples():
            for d in destinations.itertuples():
                distance = abs(o.lat - d.lat) * 111_000 + abs(o.lon - d.lon) * 71_000
                rows.append({"origin_id": o.ID, "destination_id": d.ID, "distance_m": distance})
        return pd.DataFrame(rows)


constant_matrix = ConstantRouter().prepare().route_many(source_handle.frame(), target_handle.frame())
manhattan_matrix = ManhattanRouter().prepare().route_many(source_handle.frame(), target_handle.frame())

assert len(constant_matrix) == len(manhattan_matrix) == 9
assert constant_matrix["distance_m"].sum() != manhattan_matrix["distance_m"].sum()
assert source_handle is target_handle

{
    "constant_rows": len(constant_matrix),
    "manhattan_rows": len(manhattan_matrix),
    "data_handle_still_reused": source_handle is target_handle,
}


## Pandas/Polars Transparency

In [ ]:
try:
    import polars as pl
except ImportError:
    pl = None


def row_count(frame) -> int:
    return frame.height if hasattr(frame, "height") else len(frame)

pandas_count = row_count(shared_locations)
polars_count = row_count(pl.from_pandas(shared_locations)) if pl is not None else None

assert pandas_count == 3
if polars_count is not None:
    assert polars_count == pandas_count

{
    "pandas_rows": pandas_count,
    "polars_rows": polars_count,
    "polars_available": pl is not None,
}


## Result

The tests above assert the core architectural promise: identity is explicit, data reuse is handled by the registry, artifacts remain implementation-light, and routers operate behind a strategy interface.
